In [2]:
import pandas as pd
import lightkurve as lk

In [7]:
# Filter the results to prefer SPOC, then TESS-SPOC, then QLP. If none of these exist, return all available data
def filter_spoc(group):
    if (group['author'] == 'SPOC').any():
        return group[group['author'] == 'SPOC']
    elif (group['author'] == 'TESS-SPOC').any():
        return group[group['author'] == 'TESS-SPOC']
    elif (group['author'] == 'QLP').any():
        return group[group['author'] == 'QLP']
    else:
        return group

def grab_target_table(ticid):    
    search_result = lk.search_lightcurve(f'TIC {ticid}')
    df = search_result.table.to_pandas()
    pd.set_option('display.max_rows', None)  # Show all rows
    df = df[['author', 'mission', 'exptime']] # keep only relevant columns
    
    df = df[df['author'] != 'DIAMANTE'] # remove DIAMANTE rows

    filtered_df = df.groupby('mission', group_keys=False)[['author', 'mission', 'exptime']].apply(filter_spoc).reset_index(drop=True)
    return filtered_df.sort_values(by='mission') # filtered data, sorted by TESS sector

def construct_tess_table(toi_ids, tic_ids, save=False):
    all_tables = []
    for toi, tic in zip(toi_ids, tic_ids):
        table = grab_target_table(tic)
        table['TOI'] = toi
        all_tables.append(table)
    master_table = pd.concat(all_tables, ignore_index=True)
    master_table = master_table[['TOI', 'mission', 'exptime', 'author']]

    # merge rows with the same cadence, different sectors, and the same source,
    master_table = master_table.groupby(['TOI', 'exptime', 'author'], as_index=False).agg({'mission': lambda x: ', '.join(x.astype(str))})

    # order the table by TOI (lowest to highest), then by cadence (highest to lowest)
    master_table = master_table.sort_values(by=['TOI', 'exptime', 'author'], ascending=[True, False, True], ignore_index=True)

    # replace 2nd-nth occurrence of a given TOI with '---'
    master_table['TOI'] = master_table.groupby('TOI')['TOI'].transform(lambda x: x.where(x.index == x.index[0], '---'))

    # reformat the table for LaTeX output
    master_table['mission'] = master_table['mission'].str.replace('TESS Sector ', '')
    master_table['exptime'] = master_table['exptime'].astype(int)

    master_table = master_table[['TOI', 'mission', 'exptime', 'author']]
    master_table.columns = ['TOI', 'Sector', 'Cadence (s)', 'Source']

    # convert the table to LaTeX format and write to a .tex file
    latex_table = master_table.to_latex(index=False, caption=r'Summary of \textit{TESS} Observations', label='tab:tess', column_format='cccc')

    if save:
        with open('tess_table.tex', 'w') as f:
            f.write(latex_table)

    return master_table

In [9]:
tic_ids = ['371864043', '34297761', '191202679', '312811620', '284206913', '452938965', '320281287', '368734712', '444558604', '302381397'
, '400103802', '313194972', '193765661', '86898676', '67444896', '345193111', '371573539', '190822775', '67478724', '197743152'
, '257207557', '326475995', '63718617', '95191643', '125563127', '242674266', '125695940', '352409708', '285592400', '159084486']
# tic_ids = ['371864043', '34297761']
toi_ids = ['3041', '3365', '3601', '3788', '3972', '3988', '3998', '4009', '4079', '4088', '4140', '4144', '5236', '5432', '5479', '5925',
            '6148', '6166', '6171', '6191', '6184', '6208', '6334', '6417', '6443', '7219', '7266', '7404', '7425', '7574']
tess_table = construct_tess_table(toi_ids, tic_ids, save=True)

In [10]:
tess_table

,TOI,Sector,Cadence (s),Source
0,3041,"10, 11",1800,TESS-SPOC
1,---,"36, 37, 38",600,TESS-SPOC
2,---,"63, 65, 90",120,SPOC
3,3365,09,1800,TESS-SPOC
4,---,"35, 36",600,TESS-SPOC
5,---,"62, 63, 89, 90, 99",120,SPOC
6,3601,17,1800,QLP
7,---,"57, 84",120,SPOC
8,3788,20,1800,TESS-SPOC
9,---,"60, 73",120,SPOC
